<a href="https://colab.research.google.com/github/gouripgirish73-oss/flask/blob/main/machinelearning1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [26]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load the selected diabetes dataset
diabetes_df = pd.read_csv('/content/diabetes_prediction_dataset.csv')

# Display basic information
print("Dataset Shape:", diabetes_df.shape)
diabetes_df.info()

Dataset Shape: (100000, 9)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 9 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   gender               100000 non-null  object 
 1   age                  100000 non-null  float64
 2   hypertension         100000 non-null  int64  
 3   heart_disease        100000 non-null  int64  
 4   smoking_history      100000 non-null  object 
 5   bmi                  100000 non-null  float64
 6   HbA1c_level          100000 non-null  float64
 7   blood_glucose_level  100000 non-null  int64  
 8   diabetes             100000 non-null  int64  
dtypes: float64(3), int64(4), object(2)
memory usage: 6.9+ MB


In [27]:
# Display the first few rows of the dataset
display(diabetes_df.head())

,gender,age,hypertension,heart_disease,smoking_history,bmi,HbA1c_level,blood_glucose_level,diabetes
0,Female,80.0,0,1,never,25.19,6.6,140,0
1,Female,54.0,0,0,No Info,27.32,6.6,80,0
2,Male,28.0,0,0,never,27.32,5.7,158,0
3,Female,36.0,0,0,current,23.45,5.0,155,0
4,Male,76.0,1,1,current,20.14,4.8,155,0


## Model Training and Comparison
We will train three different classifiers:
1. Logistic Regression (Baseline)
2. Random Forest Classifier
3. Gradient Boosting Classifier

We will then evaluate them on the test set and plot their ROC curves.

In [28]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report, roc_auc_score, accuracy_score, f1_score

# Initialize models
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=5, random_state=42)
}

# Dictionary to store results
results = {}

# Train and evaluate each model
for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train_scaled, y_train)
    preds = model.predict(X_test_scaled)
    probs = model.predict_proba(X_test_scaled)[:, 1]

    results[name] = {
        "Accuracy": accuracy_score(y_test, preds),
        "F1-Score": f1_score(y_test, preds),
        "ROC-AUC": roc_auc_score(y_test, probs)
    }

# Convert results to DataFrame for comparison
results_df = pd.DataFrame(results).T
display(results_df)

Training Logistic Regression...
Training Random Forest...
Training Gradient Boosting...


,Accuracy,F1-Score,ROC-AUC
Logistic Regression,0.816667,0.666667,0.858793
Random Forest,0.850000,0.727273,0.894095
Gradient Boosting,0.816667,0.685714,0.843389


## Exploratory Data Analysis & Understanding the Data
Let's check the distribution of categorical features, look for missing values, and check the target variable balance.

In [29]:
# Check unique values in categorical columns
print("Gender values:\n", diabetes_df['gender'].value_counts())
print("\nSmoking History values:\n", diabetes_df['smoking_history'].value_counts())
print("\nTarget Class Balance (Diabetes):\n", diabetes_df['diabetes'].value_counts(normalize=True))

Gender values:
 gender
Female    58552
Male      41430
Other        18
Name: count, dtype: int64

Smoking History values:
 smoking_history
No Info        35816
never          35095
former          9352
current         9286
not current     6447
ever            4004
Name: count, dtype: int64

Target Class Balance (Diabetes):
 diabetes
0    0.915
1    0.085
Name: proportion, dtype: float64


## Data Preprocessing
1. Filter out anomalous categories if any (e.g., 'Other' in gender if negligible).
2. Encode categorical columns (`gender` and `smoking_history`) using One-Hot Encoding.
3. Split the data into features (X) and target (y).
4. Train-test split and scale the features.

In [30]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Filter out 'Other' from gender as it is very small
df_cleaned = diabetes_df[diabetes_df['gender'] != 'Other'].copy()

# One-Hot Encoding categorical variables
df_encoded = pd.get_dummies(df_cleaned, columns=['gender', 'smoking_history'], drop_first=True)

# Separate features and target
X = df_encoded.drop(columns=['diabetes'])
y = df_encoded['diabetes']

# Train-Test Split (stratified due to class imbalance)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Preprocessing completed successfully!")
print("Training Set Shape:", X_train_scaled.shape)
print("Testing Set Shape:", X_test_scaled.shape)

Preprocessing completed successfully!
Training Set Shape: (79985, 12)
Testing Set Shape: (19997, 12)
